# 🎓 NB2 — QLoRA Fine-Tuning (v3)

**Prerequisites**: NB1 must have completed and saved data to `/kaggle/working/data_v1_1/`
  (or attach its output dataset / set DATA_DIR). Shared `src/` + `config/` must be available.
**Output**: `/kaggle/working/output/best_adapter/`
**Run order**: NB1 → NB2 → NB3

## Section 0 — Environment Setup (no restart required)

In [ ]:
# ============================================================================
# 0A. INSTALL (idempotent — no version disruption)
# ============================================================================
import subprocess, sys, importlib

print("Installing / verifying pinned dependencies …")
# Do NOT upgrade huggingface_hub or scikit-learn to avoid environment drift
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "bitsandbytes>=0.46.1", "peft", "accelerate",
], stdout=subprocess.DEVNULL)

importlib.invalidate_caches()

# Record package versions for train_meta.json
ENV_VERSIONS = {}
for _pkg in ["torch", "transformers", "peft", "accelerate", "bitsandbytes", "huggingface_hub"]:
    try:
        _m = importlib.import_module(_pkg)
        ENV_VERSIONS[_pkg] = getattr(_m, "__version__", "unknown")
        print(f"  {_pkg:20s}  {ENV_VERSIONS[_pkg]}")
    except Exception as _e:
        ENV_VERSIONS[_pkg] = f"error: {_e}"
        print(f"  {_pkg:20s}  ERROR: {_e}")


In [ ]:
# ============================================================================
# 0B. PREFLIGHT — must all PASS before proceeding
# ============================================================================
import os, sys, torch
from packaging.version import Version

print("── Running Preflight Verification ──")

# 1. CUDA
if not torch.cuda.is_available():
    raise RuntimeError("PREFLIGHT FAILED: CUDA not visible. Select a GPU accelerator in Kaggle.")
print(f"[PASS] CUDA: {torch.cuda.get_device_name(0)}")

# 2. bitsandbytes >= 0.46.1
try:
    import bitsandbytes as _bnb
    _bnb_ver = getattr(_bnb, "__version__", "0.0.0")
    if Version(_bnb_ver) < Version("0.46.1"):
        raise RuntimeError(
            f"bitsandbytes {_bnb_ver} < 0.46.1. Run: pip install -U bitsandbytes>=0.46.1"
        )
    print(f"[PASS] bitsandbytes {_bnb_ver}")
except Exception as _e:
    if "register" in str(_e) or "Duplicate" in str(_e):
        raise RuntimeError(
            f"PREFLIGHT FAILED: bitsandbytes duplicate registration / symbol error: {_e}\n"
            "FIX: Restart Kaggle notebook kernel (Kernel -> Restart) and run from top without re-importing bnb."
        ) from _e
    raise RuntimeError(f"PREFLIGHT FAILED: bitsandbytes error: {_e}") from _e

# 3. Real forward and loss.backward() with gradient checkpointing enabled
print("Verifying 4-bit load, LoRA, gradient checkpointing, and real loss.backward() …")
try:
    from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
    from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, TaskType

    _tok_test = AutoTokenizer.from_pretrained("Qwen/Qwen3-0.6B", trust_remote_code=True)
    if _tok_test.pad_token is None:
        _tok_test.pad_token = _tok_test.eos_token

    _bnb_test = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
    )

    try:
        _m_test = AutoModelForCausalLM.from_pretrained(
            "Qwen/Qwen3-0.6B", quantization_config=_bnb_test,
            device_map="auto", trust_remote_code=True, dtype=torch.float16,
        )
    except TypeError:
        _m_test = AutoModelForCausalLM.from_pretrained(
            "Qwen/Qwen3-0.6B", quantization_config=_bnb_test,
            device_map="auto", trust_remote_code=True, torch_dtype=torch.float16,
        )

    if hasattr(_m_test.config, "enable_thinking"):
        _m_test.config.enable_thinking = False

    _m_test.config.use_cache = False
    _m_test = prepare_model_for_kbit_training(_m_test, use_gradient_checkpointing=True)
    _m_test.enable_input_require_grads()
    _m_test = get_peft_model(_m_test, LoraConfig(
        r=8, lora_alpha=16, bias="none",
        task_type=TaskType.CAUSAL_LM, target_modules=["q_proj", "v_proj"],
    ))

    _dummy_text = "int main() { return 0; }"
    _dummy_inputs = _tok_test(_dummy_text, return_tensors="pt").to(_m_test.device)
    _out = _m_test(**_dummy_inputs, labels=_dummy_inputs["input_ids"])
    _loss = _out.loss

    if not torch.isfinite(_loss):
        raise RuntimeError(f"Dummy loss is not finite: {_loss.item()}")

    # Real backward pass with gradient checkpointing
    _loss.backward()

    _trainable_grads = [p.grad for p in _m_test.parameters() if p.requires_grad]
    if not _trainable_grads or all(g is None for g in _trainable_grads):
        raise RuntimeError("Real backward pass failed: no gradients computed on trainable parameters!")
    if any(not torch.isfinite(g).all() for g in _trainable_grads if g is not None):
        raise RuntimeError("Real backward pass produced non-finite (NaN/Inf) gradients!")

    print(f"[PASS] 4-bit load + LoRA + grad checkpointing + real backward() successful (loss={_loss.item():.4f})")
    del _m_test, _tok_test, _dummy_inputs, _out, _loss, _trainable_grads
    import gc; gc.collect(); torch.cuda.empty_cache()
except Exception as _e:
    raise RuntimeError(
        f"PREFLIGHT FAILED on backward pass: {_e}\n"
        "FIX: Restart Kaggle notebook kernel and ensure bitsandbytes>=0.46.1 and transformers are consistent."
    ) from _e

print("\nAll preflight checks passed.\n")


In [ ]:
# ============================================================================
# 0C. ALL IMPORTS & SILENCED TELEMETRY
# ============================================================================
import os, sys, json, re, random, logging, warnings, gc, inspect, pathlib, hashlib, time
import dataclasses as _dc
from dataclasses import dataclass, field
from typing import Optional, List, Dict, Any
from collections import Counter, defaultdict

import torch
import numpy as np
from tqdm.auto import tqdm

from transformers import (
    AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig,
    Trainer, TrainingArguments, TrainerCallback,
)
from peft import (
    LoraConfig, get_peft_model, prepare_model_for_kbit_training,
    TaskType, PeftModel,
)

warnings.filterwarnings("ignore", category=FutureWarning)
logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")
logger = logging.getLogger("nb2_train")

# Silence noisy HTTP and telemetry loggers (drowns out training output)
for _noisy_logger in ["httpx", "huggingface_hub", "urllib3", "urllib3.connectionpool"]:
    logging.getLogger(_noisy_logger).setLevel(logging.WARNING)


In [ ]:
# ============================================================================
# 0C-bis. SHARED CODE (src/ — one copy of schema, prompt and paths; README §5)
# ============================================================================
import os, sys, glob as _glob, shutil, importlib

# Clear stale cached modules from previous cell attempts
for _m in list(sys.modules.keys()):
    if _m == "src" or _m.startswith("src."):
        sys.modules.pop(_m, None)
importlib.invalidate_caches()

def _setup_project_root() -> str:
    """Locate or stage src/ + config/ layout for import."""
    pkg_dir = "/kaggle/working/cpp_grader_pkg" if os.path.isdir("/kaggle/working") else os.path.join(os.getcwd(), "cpp_grader_pkg")
    os.makedirs(os.path.join(pkg_dir, "src"), exist_ok=True)
    os.makedirs(os.path.join(pkg_dir, "config"), exist_ok=True)

    # Check candidate roots
    cands = []
    if os.environ.get("CPP_GRADER_ROOT") and os.path.isdir(os.environ["CPP_GRADER_ROOT"]):
        cands.append(os.environ["CPP_GRADER_ROOT"])
    cands += [os.getcwd(), os.path.join(os.getcwd(), "cpp-grader-code"), "/kaggle/working"]
    cands += sorted({os.path.dirname(os.path.dirname(p))
                     for p in _glob.glob("/kaggle/input/**/src/schema.py", recursive=True)})
    cands += sorted({os.path.dirname(p)
                     for p in _glob.glob("/kaggle/input/**/schema.py", recursive=True)})

    for root in cands:
        if os.path.abspath(root) == os.path.abspath(pkg_dir):
            continue
        s_path = os.path.join(root, "src", "schema.py") if os.path.isfile(os.path.join(root, "src", "schema.py")) else os.path.join(root, "schema.py")
        v_path = os.path.join(root, "config", "schema_version.txt") if os.path.isfile(os.path.join(root, "config", "schema_version.txt")) else os.path.join(root, "schema_version.txt")
        if os.path.isfile(s_path) and os.path.isfile(v_path):
            src_source = os.path.join(root, "src") if os.path.isdir(os.path.join(root, "src")) else root
            cfg_source = os.path.join(root, "config") if os.path.isdir(os.path.join(root, "config")) else root

            # Copy all python source modules (prompts.py, schema.py, parse.py, etc.)
            for fn in os.listdir(src_source):
                if fn.endswith(".py"):
                    src_f = os.path.join(src_source, fn)
                    dst_f = os.path.join(pkg_dir, "src", fn)
                    if os.path.abspath(src_f) != os.path.abspath(dst_f):
                        shutil.copy2(src_f, dst_f)

            # Copy schema_version.txt
            if os.path.isfile(os.path.join(cfg_source, "schema_version.txt")):
                dst_v = os.path.join(pkg_dir, "config", "schema_version.txt")
                if os.path.abspath(os.path.join(cfg_source, "schema_version.txt")) != os.path.abspath(dst_v):
                    shutil.copy2(os.path.join(cfg_source, "schema_version.txt"), dst_v)
            return pkg_dir

    # If already staged in pkg_dir
    if (os.path.isfile(os.path.join(pkg_dir, "src", "schema.py"))
            and os.path.isfile(os.path.join(pkg_dir, "config", "schema_version.txt"))):
        return pkg_dir

    raise RuntimeError(
        f"Shared src/ + config/ not found in {cands}. Attach cpp-grader-code or set CPP_GRADER_ROOT."
    )

PROJECT_ROOT = _setup_project_root()
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

print(f"Shared code: {PROJECT_ROOT}")

from src.schema import SCHEMA_VERSION, validate_grading_output
from src.prompts import SYSTEM_PROMPT
from src.paths import resolve_data_dir


In [ ]:
# ============================================================================
# 0D. CONFIG
# ============================================================================

# Dynamic data directory resolved via src.paths.resolve_data_dir()

@dataclass
class TrainConfig:
    seed:               int   = 42

    # Model
    student_model_id:   str   = "Qwen/Qwen3-4B"

    # Quantisation
    bnb_4bit_quant_type:        str   = "nf4"
    bnb_4bit_compute_dtype:     str   = "float16"
    bnb_4bit_use_double_quant:  bool  = True

    # Sequence length: set dynamically from measured max length (not p99)
    max_seq_len:        int   = 2304  # Set from Phase 0 token audit (p99 prompt 894 + response 156 ~ 1050)
    train_target_format: str   = os.environ.get("TRAIN_TARGET_FORMAT", "reasoning")  # "reasoning" (Arm A2) | "direct" (Arm A1)
    experiment_arm:      str   = os.environ.get("EXPERIMENT_ARM", "A2")  # "A1" | "A2"

    # LoRA
    lora_r:             int   = 16
    lora_alpha:         int   = 32
    lora_dropout:       float = 0.05
    lora_target_modules: list = field(default_factory=lambda:
        ["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"])

    # Training: default epochs=1 until measured epoch time is known
    num_train_epochs:             int   = 1
    max_train_hours:              float = 7.5   # Clean stop limit before Kaggle timeout
    per_device_train_batch_size:  int   = 1
    per_device_eval_batch_size:   int   = 1
    gradient_accumulation_steps:  int   = 16
    learning_rate:                float = 2e-4
    lr_scheduler_type:            str   = "cosine"
    warmup_ratio:                 float = 0.03
    weight_decay:                 float = 0.01
    optim:                        str   = "paged_adamw_8bit"
    fp16:                         bool  = True
    bf16:                         bool  = False
    gradient_checkpointing:       bool  = True

    # Evaluation during training
    val_subset_size:              int   = 120   # Evaluate on subset of at most 120 examples
    eval_steps:                   int   = 25
    save_steps:                   int   = 25
    logging_steps:                int   = 1     # Step-by-step logging to verify warmup
    max_grad_norm:                float = 0.3

    # Smoke
    smoke_test:         bool  = False
    smoke_max_steps:    int   = 20

    # Paths
    data_dir:           str   = ""
    output_dir:         str   = "output"
    checkpoints_root:   str   = "checkpoints"


CFG = TrainConfig()
SMOKE_TEST = CFG.smoke_test

IS_KAGGLE = os.path.exists("/kaggle/working")
BASE_DIR  = "/kaggle/working" if IS_KAGGLE else "."

CFG.data_dir         = resolve_data_dir()
CFG.output_dir       = os.path.join(BASE_DIR, "output")
CFG.checkpoints_root = os.path.join(BASE_DIR, "checkpoints")
os.makedirs(CFG.output_dir,       exist_ok=True)
os.makedirs(CFG.checkpoints_root, exist_ok=True)

BEST_ADAPTER_PATH = os.path.join(CFG.output_dir, "best_adapter")

def seed_everything(seed: int) -> None:
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)

seed_everything(CFG.seed)

print(f"Data dir         = {CFG.data_dir}")
print(f"SCHEMA_VERSION   = {SCHEMA_VERSION}")
print(f"SMOKE_TEST       = {SMOKE_TEST}")
print(f"Student model    = {CFG.student_model_id}")
print(f"Configured Epochs = {CFG.num_train_epochs} (max_train_hours={CFG.max_train_hours}h)")


## Section 1 — Verify NB1 Outputs

In [ ]:
# ============================================================================
# 1. VERIFY NB1 DATA, SCHEMA, DISJOINT SPLITS & MEASURE MAX_SEQ_LEN
# ============================================================================

# Check available SFT files (v2 or v1.1 fallback)
sft_train_name = "train_sft_v2.jsonl" if getattr(CFG, "train_target_format", "reasoning") == "reasoning" else "train_sft_v2_direct.jsonl"
if not os.path.exists(os.path.join(CFG.data_dir, sft_train_name)):
    sft_train_name = "train_sft.jsonl"

sft_val_name = "val_unseen_sft.jsonl" if os.path.exists(os.path.join(CFG.data_dir, "val_unseen_sft.jsonl")) else "val_sft.jsonl"

for fname in [sft_train_name, sft_val_name, "data_manifest.json"]:
    fpath = os.path.join(CFG.data_dir, fname)
    assert os.path.exists(fpath), f"Missing {fpath} — run NB1 (nb1_data.py) first."

# Check schema version matches
with open(os.path.join(CFG.data_dir, "data_manifest.json")) as f:
    _manifest = json.load(f)
assert _manifest.get("schema_version") == SCHEMA_VERSION, (
    f"Schema version mismatch: NB1 produced {_manifest.get('schema_version')!r}, "
    f"NB2 expects {SCHEMA_VERSION!r}. Re-run NB1."
)
print(f"[PASS] NB1 manifest schema_version = {SCHEMA_VERSION}")

# Assert and print distinct program_id values per split and check cross-split disjointness
split_progs = {}
for split_name in ["train", "val_seen", "val_unseen", "test_seen", "test_unseen_synth", "test_unseen_real_tasks", "val", "test_unseen"]:
    split_file = os.path.join(CFG.data_dir, f"{split_name}.jsonl")
    if not os.path.isfile(split_file):
        continue
    p_set = set()
    with open(split_file, "r", encoding="utf-8") as f:
        for line_no, line in enumerate(f, 1):
            if not line.strip(): continue
            row = json.loads(line)
            if "program_id" not in row or not row["program_id"]:
                raise RuntimeError(
                    f"Row {line_no} in {split_name}.jsonl is missing 'program_id'. "
                    "The data is in the old format. Stop and re-run NB1 with v4 generation."
                )
            p_set.add(row["program_id"])
    split_progs[split_name] = p_set
    print(f"  Split '{split_name:11s}': {len(p_set):4d} distinct program_id values")

# Cross-split disjointness check
splits_to_check = list(split_progs.keys())
for i in range(len(splits_to_check)):
    for j in range(i + 1, len(splits_to_check)):
        s1, s2 = splits_to_check[i], splits_to_check[j]
        overlap = split_progs[s1].intersection(split_progs[s2])
        assert len(overlap) == 0, (
            f"DATA LEAKAGE DETECTED: {len(overlap)} overlapping program_id values between {s1} and {s2}: {list(overlap)[:5]}"
        )
print("[PASS] 0 program_id overlap across all splits (data isolation confirmed).")

# Measure exact prompt+target lengths across train and val SFT examples
print("\nMeasuring sequence lengths across SFT datasets …")
_tok_measure = AutoTokenizer.from_pretrained(CFG.student_model_id, trust_remote_code=True)
lengths_by_question = defaultdict(list)
all_measured_lengths = []

for sft_file in ["train_sft.jsonl", "val_sft.jsonl"]:
    with open(os.path.join(CFG.data_dir, sft_file), "r", encoding="utf-8") as f:
        for line in f:
            if not line.strip(): continue
            row = json.loads(line)
            qid = row.get("question_id", "unknown")
            try:
                text = _tok_measure.apply_chat_template(
                    row["messages"], tokenize=False, add_generation_prompt=False, enable_thinking=False
                )
            except TypeError:
                text = _tok_measure.apply_chat_template(
                    row["messages"], tokenize=False, add_generation_prompt=False
                )
            tok_len = len(_tok_measure(text, add_special_tokens=False).input_ids)
            lengths_by_question[qid].append(tok_len)
            all_measured_lengths.append(tok_len)

print("\nLength distribution per question (min / med / p95 / max):")
for qid in sorted(lengths_by_question.keys()):
    lens = lengths_by_question[qid]
    print(f"  {qid:22s}: min={min(lens):4d} | med={int(np.median(lens)):4d} | p95={int(np.percentile(lens, 95)):4d} | max={max(lens):4d}")

abs_max_len = max(all_measured_lengths)
rounded_max = int(np.ceil(abs_max_len / 64.0) * 64)
CFG.max_seq_len = rounded_max
print(f"\nMaximum measured sequence length: {abs_max_len}")
print(f"CFG.max_seq_len set to: {CFG.max_seq_len} (rounded up to multiple of 64; guarantees 0 dropped)")
del _tok_measure


## Section 2 — Custom Dataset

In [ ]:
# ============================================================================
# 2A. SYSTEM PROMPT + VALIDATOR
# ============================================================================
# Imported from src.prompts / src.schema in section 0C-bis (one copy, README §5).
# The SFT JSONL already contains the full prompt built by NB1 with the same code.

train_sft_file = os.path.join(CFG.data_dir, sft_train_name)
with open(train_sft_file, encoding="utf-8") as _f:
    _first_sys = json.loads(_f.readline())["messages"][0]["content"]
assert _first_sys == SYSTEM_PROMPT, (
    f"{sft_train_name} system prompt differs from src.prompts.SYSTEM_PROMPT — "
    "re-run NB1 with the current src/."
)
print("[PASS] SFT system prompt matches src.prompts.SYSTEM_PROMPT")

In [ ]:
# ============================================================================
# 2B. CUSTOM DATASET — completion-only masking
# ============================================================================

RESPONSE_TEMPLATE = "<|im_start|>assistant\n"

class GraderSFTDataset(torch.utils.data.Dataset):
    """
    Reads SFT JSONL, tokenizes, applies completion-only masking.
    Asserts 0 dropped examples because max_seq_len covers the maximum measured sequence.
    """

    def __init__(self, jsonl_path: str, tokenizer, cfg: TrainConfig):
        self.cfg       = cfg
        self.tokenizer = tokenizer
        self.examples  = []
        self._resp_ids = tokenizer.encode(RESPONSE_TEMPLATE, add_special_tokens=False)
        self._dropped  = 0
        self.dropped_log = []
        self._load(jsonl_path)

    def _load(self, path: str) -> None:
        rows = []
        with open(path, "r", encoding="utf-8") as f:
            for line in f:
                if line.strip():
                    rows.append(json.loads(line.strip()))
        logger.info(f"Read {len(rows)} raw examples from {path}")

        for row in tqdm(rows, desc="Tokenising", leave=False):
            msgs = row["messages"]
            try:
                try:
                    text = self.tokenizer.apply_chat_template(
                        msgs, tokenize=False, add_generation_prompt=False,
                        enable_thinking=False,
                    )
                except TypeError:
                    text = self.tokenizer.apply_chat_template(
                        msgs, tokenize=False, add_generation_prompt=False,
                    )

                ids = self.tokenizer(
                    text, add_special_tokens=False
                ).input_ids

                # Hard drop if exceeds max_seq_len
                if len(ids) > self.cfg.max_seq_len:
                    self._dropped += 1
                    self.dropped_log.append({"sample_id": row.get("sample_id", ""),
                                             "reason": "exceeds_max_seq_len",
                                             "n_tokens": len(ids), "max_seq_len": self.cfg.max_seq_len})
                    continue

                # Find response template
                rlen  = len(self._resp_ids)
                start = None
                for i in range(len(ids) - rlen + 1):
                    if ids[i:i + rlen] == self._resp_ids:
                        start = i + rlen
                        break

                if start is None:
                    raise ValueError(
                        f"Response template {RESPONSE_TEMPLATE!r} not found in tokenised output.\n"
                        f"First 200 tokens: {self.tokenizer.decode(ids[:200])!r}"
                    )

                prompt_msgs = msgs[:-1]
                try:
                    prompt_text = self.tokenizer.apply_chat_template(
                        prompt_msgs, tokenize=False, add_generation_prompt=True,
                        enable_thinking=False,
                    )
                except TypeError:
                    prompt_text = self.tokenizer.apply_chat_template(
                        prompt_msgs, tokenize=False, add_generation_prompt=True,
                    )
                prompt_ids = self.tokenizer(
                    prompt_text, add_special_tokens=False
                ).input_ids
                assert ids[:len(prompt_ids)] == prompt_ids, (
                    "Prompt is not a prefix of full token sequence — "
                    "chat template may differ between prompt-only and full-message."
                )
                assert start <= len(prompt_ids), "Response template found after the prompt end."

                # Exact completion-only mask
                labels = [-100] * len(prompt_ids) + ids[len(prompt_ids):]
                assert any(l != -100 for l in labels), \
                    "All label tokens are masked — response template mismatch."

                self.examples.append({
                    "input_ids":      torch.tensor(ids,    dtype=torch.long),
                    "attention_mask": torch.tensor([1]*len(ids), dtype=torch.long),
                    "labels":         torch.tensor(labels, dtype=torch.long),
                    "sample_id":      row.get("sample_id", ""),
                    "question_id":    row.get("question_id", ""),
                })

            except Exception as e:
                raise RuntimeError(f"Tokenisation/masking failed for {row.get('sample_id', '?')}: {e}") from e

        logger.info(f"Kept {len(self.examples)} / {len(rows)} (dropped {self._dropped})")

    def __len__(self):
        return len(self.examples)
    def __getitem__(self, idx):
        ex = self.examples[idx]
        return {
            "input_ids":      ex["input_ids"],
            "attention_mask": ex["attention_mask"],
            "labels":         ex["labels"],
        }


In [ ]:
# ============================================================================
# 2C. PADDING COLLATOR
# ============================================================================

class PaddingCollator:
    def __init__(self, pad_id: int):
        self.pad_id = pad_id

    def __call__(self, features: list) -> dict:
        max_len = max(f["input_ids"].shape[0] for f in features)
        iids, amask, lbls = [], [], []
        for f in features:
            pad = max_len - f["input_ids"].shape[0]
            iids.append(torch.cat([f["input_ids"],
                                   torch.full((pad,), self.pad_id, dtype=torch.long)]))
            amask.append(torch.cat([f["attention_mask"],
                                    torch.zeros(pad, dtype=torch.long)]))
            lbls.append(torch.cat([f["labels"],
                                   torch.full((pad,), -100, dtype=torch.long)]))
        return {
            "input_ids":      torch.stack(iids),
            "attention_mask": torch.stack(amask),
            "labels":         torch.stack(lbls),
        }

## Section 3 — Load Model + Apply LoRA

In [ ]:
# ============================================================================
# 3A. LOAD TOKENIZER
# ============================================================================

logger.info(f"Loading tokenizer: {CFG.student_model_id}")
tokenizer = AutoTokenizer.from_pretrained(
    CFG.student_model_id, trust_remote_code=True, padding_side="right"
)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

In [ ]:
# ============================================================================
# 3B. BUILD DATASETS & EVALUATION SUBSET
# ============================================================================

train_dataset = GraderSFTDataset(
    os.path.join(CFG.data_dir, "train_sft.jsonl"), tokenizer, CFG
)
val_dataset_full = GraderSFTDataset(
    os.path.join(CFG.data_dir, "val_sft.jsonl"),   tokenizer, CFG
)

assert len(train_dataset) > 0, "Training dataset is empty after tokenisation."
assert len(val_dataset_full) > 0, "Validation dataset is empty."

# Assert zero dropped examples as required by prompt
assert train_dataset._dropped == 0, f"Expected 0 dropped in train, but dropped {train_dataset._dropped}!"
assert val_dataset_full._dropped == 0, f"Expected 0 dropped in val, but dropped {val_dataset_full._dropped}!"

print(f"\n[PASS] Train: {len(train_dataset)} examples (dropped: {train_dataset._dropped})")
print(f"[PASS] Val:   {len(val_dataset_full)} examples (dropped: {val_dataset_full._dropped})")

# Subset validation dataset for training evaluations to keep evaluation fast
if len(val_dataset_full) > CFG.val_subset_size:
    rng = np.random.RandomState(CFG.seed)
    sub_indices = rng.choice(len(val_dataset_full), size=CFG.val_subset_size, replace=False)
    sub_indices.sort()
    eval_dataset = torch.utils.data.Subset(val_dataset_full, sub_indices)
    print(f"Val subset for training evaluations: {len(eval_dataset)} examples (out of {len(val_dataset_full)})")
else:
    eval_dataset = val_dataset_full

collator = PaddingCollator(pad_id=tokenizer.pad_token_id or tokenizer.eos_token_id)

_s = train_dataset[0]
_n_resp = (_s["labels"] != -100).sum().item()
print(f"Sample: input_ids={_s['input_ids'].shape}  non-masked labels={_n_resp}")
assert _n_resp > 0, "No non-masked label tokens — completion-only masking failed."


In [ ]:
# ============================================================================
# 3C. LOAD MODEL IN 4-BIT
# ============================================================================

logger.info(f"Loading student model: {CFG.student_model_id}")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type=CFG.bnb_4bit_quant_type,
    bnb_4bit_compute_dtype=getattr(torch, CFG.bnb_4bit_compute_dtype),
    bnb_4bit_use_double_quant=CFG.bnb_4bit_use_double_quant,
)

try:
    model = AutoModelForCausalLM.from_pretrained(
        CFG.student_model_id,
        quantization_config=bnb_config,
        device_map="auto",
        trust_remote_code=True,
        dtype=torch.float16,
        attn_implementation="sdpa",
    )
except TypeError:
    try:
        model = AutoModelForCausalLM.from_pretrained(
            CFG.student_model_id,
            quantization_config=bnb_config,
            device_map="auto",
            trust_remote_code=True,
            torch_dtype=torch.float16,
            attn_implementation="sdpa",
        )
    except TypeError:
        model = AutoModelForCausalLM.from_pretrained(
            CFG.student_model_id,
            quantization_config=bnb_config,
            device_map="auto",
            trust_remote_code=True,
            torch_dtype=torch.float16,
        )
except Exception as e:
    raise RuntimeError(
        f"Failed to load {CFG.student_model_id}: {e}\n"
        "No fallback model will be loaded. Fix the error above and retry."
    )

# Disable Qwen3 thinking mode (prevents <think> token embeddings causing NaN in fp16)
if hasattr(model.config, "enable_thinking"):
    model.config.enable_thinking = False

model.enable_input_require_grads()
model = prepare_model_for_kbit_training(
    model, use_gradient_checkpointing=CFG.gradient_checkpointing
)

lora_config = LoraConfig(
    r=CFG.lora_r,
    lora_alpha=CFG.lora_alpha,
    lora_dropout=CFG.lora_dropout,
    target_modules=CFG.lora_target_modules,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

## Section 4 — Training

In [ ]:
# ============================================================================
# 4A. TRAINING ARGS (warmup_steps integer, clean non-nested checkpoints)
# ============================================================================

effective_batch = CFG.per_device_train_batch_size * CFG.gradient_accumulation_steps
steps_per_epoch = int(np.ceil(len(train_dataset) / effective_batch))
total_steps = CFG.smoke_max_steps if SMOKE_TEST else (steps_per_epoch * CFG.num_train_epochs)
warmup_steps = max(1, int(CFG.warmup_ratio * total_steps))

# Unique run signature to isolate checkpoints
_run_sig = hashlib.sha256(json.dumps({
    "cfg": {k: v for k, v in _dc.asdict(CFG).items() if k not in ("data_dir", "output_dir", "checkpoints_root")},
    "total_steps": total_steps, "schema_version": SCHEMA_VERSION,
    "data_file_hashes": _manifest.get("file_hashes"),
}, sort_keys=True, default=str).encode()).hexdigest()[:12]

# Clean non-nested directory: <checkpoints_root>/run_<sig>
RUN_CHECKPOINT_DIR = os.path.join(CFG.checkpoints_root, f"run_{_run_sig}")
os.makedirs(RUN_CHECKPOINT_DIR, exist_ok=True)
print(f"Checkpoints dir = {RUN_CHECKPOINT_DIR}")

# Resume from latest checkpoint in RUN_CHECKPOINT_DIR
_ckpts = sorted(
    [d for d in os.listdir(RUN_CHECKPOINT_DIR) if d.startswith("checkpoint-")],
    key=lambda d: int(d.split("-")[-1]) if d.split("-")[-1].isdigit() else -1,
) if os.path.isdir(RUN_CHECKPOINT_DIR) else []
RESUME_FROM = os.path.join(RUN_CHECKPOINT_DIR, _ckpts[-1]) if _ckpts else None
print(f"Resume from     = {RESUME_FROM or 'None (fresh run)'}")
print(f"Total steps     = {total_steps} (warmup_steps={warmup_steps}, steps/epoch={steps_per_epoch})")

training_args = TrainingArguments(
    output_dir=RUN_CHECKPOINT_DIR,
    num_train_epochs=CFG.num_train_epochs,
    per_device_train_batch_size=CFG.per_device_train_batch_size,
    per_device_eval_batch_size=CFG.per_device_eval_batch_size,
    gradient_accumulation_steps=CFG.gradient_accumulation_steps,
    learning_rate=CFG.learning_rate,
    lr_scheduler_type=CFG.lr_scheduler_type,
    warmup_steps=warmup_steps,               # Integer warmup_steps directly passed
    weight_decay=CFG.weight_decay,
    optim=CFG.optim,
    fp16=CFG.fp16,
    bf16=CFG.bf16,
    gradient_checkpointing=CFG.gradient_checkpointing,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    eval_strategy="steps",
    eval_steps=CFG.eval_steps,
    save_strategy="steps",
    save_steps=CFG.save_steps,
    save_total_limit=3,
    logging_steps=CFG.logging_steps,          # Log step 1-5 for learning rate verification
    max_grad_norm=CFG.max_grad_norm,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    report_to="none",
    remove_unused_columns=False,
    max_steps=total_steps if SMOKE_TEST else -1,
    seed=CFG.seed,
)


In [ ]:
# ============================================================================
# 4B. NAN LOSS DETECTOR, WALL-CLOCK ETA & TIME-BUDGET CALLBACK
# ============================================================================

class TimeBudgetCallback(TrainerCallback):
    """Stops training cleanly when time budget is reached, preserving best model."""
    def __init__(self, max_hours: float):
        self.max_seconds = max_hours * 3600.0
        self.start_time = None

    def on_train_begin(self, args, state, control, **kwargs):
        self.start_time = time.time()

    def on_step_end(self, args, state, control, **kwargs):
        if self.start_time is None: return
        elapsed = time.time() - self.start_time
        if elapsed >= self.max_seconds:
            print(f"\n[TIME BUDGET] {elapsed/3600.0:.2f}h elapsed >= {self.max_seconds/3600.0:.2f}h limit!")
            print("Stopping training cleanly to keep best adapter checkpoint.")
            control.should_training_stop = True



class ValUnseenGenerationCallback(TrainerCallback):
    """Evaluates on a fixed 100-sample subset of val_unseen using generated outputs.
    
    Measures marks agreement (exact match and MAE) and parse validity (format compliance),
    saving the best adapter based on generation accuracy rather than loss alone.
    """
    def __init__(self, val_jsonl_path: str, tokenizer, target_path: str, max_samples: int = 100):
        self.val_samples = []
        if os.path.isfile(val_jsonl_path):
            with open(val_jsonl_path, "r", encoding="utf-8") as f:
                for line in f:
                    if line.strip():
                        self.val_samples.append(json.loads(line.strip()))
                        if len(self.val_samples) >= max_samples:
                            break
        self.tokenizer = tokenizer
        self.target_path = target_path
        self.best_score = -float("inf")
        self.best_exact_match = 0.0

    def on_evaluate(self, args, state, control, metrics=None, **kwargs):
        if not self.val_samples:
            return
        model = kwargs.get("model")
        if model is None:
            return

        model.eval()
        n_exact = 0
        n_parse_ok = 0
        total_abs_err = 0.0
        n_eval = len(self.val_samples)

        with torch.no_grad():
            for sample in self.val_samples:
                msgs = sample.get("messages", [])[:-1]  # Exclude assistant target
                try:
                    prompt_text = self.tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True, enable_thinking=False)
                except TypeError:
                    prompt_text = self.tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
                
                inputs = self.tokenizer(prompt_text, return_tensors="pt").to(model.device)
                out_ids = model.generate(
                    **inputs,
                    max_new_tokens=400,
                    do_sample=False,
                    pad_token_id=self.tokenizer.pad_token_id or self.tokenizer.eos_token_id
                )
                gen_text = self.tokenizer.decode(out_ids[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
                
                try:
                    from src.parse import parse_and_validate_output
                    rubric = sample.get("rubric", [])
                    parsed = parse_and_validate_output(gen_text, rubric)
                    if parsed.is_valid:
                        n_parse_ok += 1
                    pred_tot = parsed.total_marks
                except Exception:
                    pred_tot = 0.0

                gold_tot = sample.get("gold_total", 0.0)
                diff = abs(pred_tot - gold_tot)
                total_abs_err += diff
                if diff < 0.01:
                    n_exact += 1

        exact_rate = n_exact / max(1, n_eval)
        parse_rate = n_parse_ok / max(1, n_eval)
        mae = total_abs_err / max(1, n_eval)
        score = exact_rate - (0.1 * mae) + (0.2 * parse_rate)

        print(f"\n[VAL_UNSEEN EVAL @ Step {state.global_step}] (N={n_eval}):")
        print(f"  • Exact Match:  {exact_rate*100:.1f}% ({n_exact}/{n_eval})")
        print(f"  • Parse Valid:  {parse_rate*100:.1f}% ({n_parse_ok}/{n_eval})")
        print(f"  • MAE:          {mae:.3f}")
        print(f"  • Gen Score:    {score:.4f} (best: {self.best_score:.4f})")

        if score > self.best_score:
            self.best_score = score
            self.best_exact_match = exact_rate
            os.makedirs(self.target_path, exist_ok=True)
            model.save_pretrained(self.target_path)
            if self.tokenizer is not None:
                self.tokenizer.save_pretrained(self.target_path)
            print(f"  → Saved new best adapter to {self.target_path} (exact_match={exact_rate*100:.1f}%)")
        model.train()

class SaveBestAdapterCallback(TrainerCallback):
    """Saves adapter to BEST_ADAPTER_PATH on every evaluation improvement."""
    def __init__(self, target_path: str):
        self.target_path = target_path
        self.best_loss = float("inf")

    def on_evaluate(self, args, state, control, metrics=None, **kwargs):
        if metrics and "eval_loss" in metrics:
            loss = metrics["eval_loss"]
            if loss < self.best_loss:
                self.best_loss = loss
                model = kwargs.get("model")
                tokenizer = kwargs.get("tokenizer")
                if model is not None:
                    os.makedirs(self.target_path, exist_ok=True)
                    model.save_pretrained(self.target_path)
                    if tokenizer is not None:
                        tokenizer.save_pretrained(self.target_path)
                    print(f"\n[BEST ADAPTER] Step {state.global_step}: new best eval_loss={loss:.4f} → saved to {self.target_path}")


class GraderTrainer(Trainer):
    """Trainer with NaN loss guard and optimizer-step throughput measurement."""

    t_step3: float = 0.0
    eval_time_during_window: float = 0.0
    last_eval_duration: float = 0.0
    timing_reported: bool = False

    def training_step(self, model, inputs, num_items_in_batch=None):
        try:
            loss = super().training_step(model, inputs, num_items_in_batch)
        except TypeError:
            loss = super().training_step(model, inputs)

        if not torch.isfinite(loss):
            raise RuntimeError(
                f"NaN/Inf loss at step {self.state.global_step}!\n"
                "Causes on T4 fp16: learning rate too high or gradient explosion.\n"
                "Fix: lower CFG.learning_rate to 1e-4, set max_grad_norm=0.1."
            )

        step = self.state.global_step
        # Start measuring pure optimizer training time at step 3
        if step == 3 and self.t_step3 == 0.0:
            self.t_step3 = time.time()
            self.eval_time_during_window = 0.0

        # Report throughput and ETA at step 23 (20 optimizer steps measured)
        if step == 23 and not self.timing_reported and self.t_step3 > 0.0:
            self.timing_reported = True
            total_wall = time.time() - self.t_step3
            pure_train_time = max(0.1, total_wall - self.eval_time_during_window)
            s_per_opt_step = pure_train_time / 20.0
            eff_batch = CFG.per_device_train_batch_size * CFG.gradient_accumulation_steps
            examples_per_sec = eff_batch / s_per_opt_step
            steps_epoch = int(np.ceil(len(self.train_dataset) / eff_batch))
            evals_per_epoch = steps_epoch // max(1, CFG.eval_steps)
            eval_sec = self.last_eval_duration if self.last_eval_duration > 0.0 else (len(self.eval_dataset) * 0.15)
            h_per_epoch = (steps_epoch * s_per_opt_step + evals_per_epoch * eval_sec) / 3600.0
            total_h = h_per_epoch * CFG.num_train_epochs

            print("\n" + "="*65)
            print("THROUGHPUT & ETA PROJECTION (measured over steps 3–23 excluding eval):")
            print(f"  • Wall-clock per optimizer step: {s_per_opt_step:.2f} s/step")
            print(f"  • Throughput:                    {examples_per_sec:.2f} examples/s")
            print(f"  • Optimizer steps / epoch:       {steps_epoch} steps")
            print(f"  • Measured eval duration:        {eval_sec:.1f} s/eval ({len(self.eval_dataset)} samples)")
            print(f"  • Projected hours per epoch:     {h_per_epoch:.2f} h")
            print(f"  • Projected total ({CFG.num_train_epochs} epoch(s)):   {total_h:.2f} h")
            if total_h > CFG.max_train_hours:
                print(f"  WARNING: Projected time ({total_h:.2f} h) exceeds budget ({CFG.max_train_hours} h)!")
                print("           TimeBudgetCallback will cleanly stop training when budget is reached.")
            print("="*65 + "\n")
        return loss

    def evaluate(self, *args, **kwargs):
        t_start = time.time()
        out = super().evaluate(*args, **kwargs)
        duration = time.time() - t_start
        self.last_eval_duration = duration
        if self.t_step3 > 0.0 and self.state.global_step <= 23:
            self.eval_time_during_window += duration
        return out

trainer = GraderTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=collator,
    callbacks=[
        TimeBudgetCallback(max_hours=CFG.max_train_hours),
        ValUnseenGenerationCallback(val_jsonl_path=os.path.join(CFG.data_dir, sft_val_name), tokenizer=tokenizer, target_path=BEST_ADAPTER_PATH, max_samples=100),
        SaveBestAdapterCallback(target_path=BEST_ADAPTER_PATH),
    ],
)
trainer.train_dataset = train_dataset

eff_batch = CFG.per_device_train_batch_size * CFG.gradient_accumulation_steps
logger.info(f"Effective batch size = {eff_batch}")


In [ ]:
# ============================================================================
# 4C. TRAIN
# ============================================================================

logger.info("Starting QLoRA fine-tuning …")
train_result = trainer.train(resume_from_checkpoint=RESUME_FROM)

In [ ]:
# ============================================================================
# 4D. SAVE ADAPTER, CONFIG & train_meta.json
# ============================================================================
import os, sys, json, logging
import dataclasses as _dc
logger = logging.getLogger("train")

if "BEST_ADAPTER_PATH" not in globals():
    _out = getattr(CFG, "output_dir", "/kaggle/working/output")
    BEST_ADAPTER_PATH = os.path.join(_out, "best_adapter")

os.makedirs(BEST_ADAPTER_PATH, exist_ok=True)
trainer.save_model(BEST_ADAPTER_PATH)
tokenizer.save_pretrained(BEST_ADAPTER_PATH)

# Fallback-safe metadata resolution
_schema_ver = globals().get("SCHEMA_VERSION", "v3.1")
_env_vers = globals().get("ENV_VERSIONS", {})
_runtime = getattr(train_result, "metrics", {}).get("train_runtime", 0.0) if "train_result" in globals() else 0.0
_n_train = len(train_dataset) if "train_dataset" in globals() else 0
_n_val_sub = len(eval_dataset) if "eval_dataset" in globals() else 0
_n_val_tot = len(val_dataset_full) if "val_dataset_full" in globals() else 0
_drop_train = getattr(train_dataset, "_dropped", 0) if "train_dataset" in globals() else 0
_drop_val = getattr(val_dataset_full, "_dropped", 0) if "val_dataset_full" in globals() else 0

train_meta = {
    "actual_model_id":   getattr(CFG, "student_model_id", "Qwen/Qwen3-4B"),
    "experiment_arm":    getattr(CFG, "experiment_arm", "A2"),
    "train_target_format": getattr(CFG, "train_target_format", "reasoning"),
    "schema_version":    _schema_ver,
    "max_seq_len":        getattr(CFG, "max_seq_len", 1024),
    "examples_used": {
        "train": _n_train,
        "val_eval_subset": _n_val_sub,
        "val_total": _n_val_tot,
    },
    "epochs_run":        getattr(trainer.state, "epoch", 1.0),
    "best_eval_loss":    getattr(trainer.state, "best_metric", None),
    "package_versions":  _env_vers,
    "seconds_trained":   _runtime,
    "dropped_train":     _drop_train,
    "dropped_val":       _drop_val,
}

meta_path = os.path.join(BEST_ADAPTER_PATH, "train_meta.json")
with open(meta_path, "w", encoding="utf-8") as f:
    json.dump(train_meta, f, indent=2, default=str)

cfg_dict = _dc.asdict(CFG) if hasattr(CFG, "__dataclass_fields__") else {}
with open(os.path.join(BEST_ADAPTER_PATH, "train_config.json"), "w", encoding="utf-8") as f:
    json.dump({
        **cfg_dict,
        **train_meta,
    }, f, indent=2, default=str)

logger.info(f"Best adapter saved → {BEST_ADAPTER_PATH}")
logger.info(f"train_meta.json     → {meta_path}")
print(f"\n[PASS] Successfully saved best adapter to {BEST_ADAPTER_PATH}")
print(f"[PASS] Successfully saved train_meta.json and train_config.json")


In [ ]:
# ============================================================================
# 4E. SAVE METRICS, PLOT & WARMUP VERIFICATION
# ============================================================================
import os, sys, json, csv as _csv

metrics = getattr(train_result, "metrics", {}) if "train_result" in globals() else {}
_out_dir = getattr(CFG, "output_dir", "/kaggle/working/output")
metrics_path = os.path.join(_out_dir, "training_metrics.csv")
if metrics:
    with open(metrics_path, "w", newline="") as f:
        w = _csv.DictWriter(f, fieldnames=list(metrics.keys()))
        w.writeheader(); w.writerow(metrics)

# Verify logged learning rates for initial steps (warm-up verification)
lr_history = [(e["step"], e["learning_rate"]) for e in trainer.state.log_history if "learning_rate" in e]
print("\nLogged learning rates for initial steps:")
for step_i, lr_val in lr_history[:5]:
    print(f"  Step {step_i}: lr = {lr_val:.6e}")

if lr_history:
    first_step, first_lr = lr_history[0]
    peak_lr = getattr(CFG, "learning_rate", 2e-4)
    assert first_lr < peak_lr, (
        f"Warm-up failed: Step {first_step} learning rate {first_lr:.6e} is not below peak {peak_lr:.6e}!"
    )
    print(f"[PASS] Warm-up confirmed: step {first_step} lr ({first_lr:.6e}) < peak lr ({peak_lr:.6e})")

history_path = os.path.join(_out_dir, "training_history.csv")
if trainer.state.log_history:
    keys = sorted({k for e in trainer.state.log_history for k in e})
    with open(history_path, "w", newline="") as f:
        w = _csv.DictWriter(f, fieldnames=keys, extrasaction="ignore")
        w.writeheader()
        for e in trainer.state.log_history:
            w.writerow(e)

# Plot curve
try:
    import matplotlib.pyplot as _plt
    _hist = trainer.state.log_history
    _train_steps = [e["step"] for e in _hist if "loss"      in e]
    _train_loss  = [e["loss"] for e in _hist if "loss"      in e]
    _eval_steps  = [e["step"] for e in _hist if "eval_loss" in e]
    _eval_loss   = [e["eval_loss"] for e in _hist if "eval_loss" in e]

    _fig, _ax = _plt.subplots(figsize=(8, 4))
    _ax.plot(_train_steps, _train_loss, label="Train loss")
    if _eval_loss:
        _ax.plot(_eval_steps, _eval_loss, label="Eval loss", marker="o")
    _ax.set_xlabel("Step"); _ax.set_ylabel("Loss")
    _ax.set_title("Training Curve"); _ax.legend()
    _plt.tight_layout()
    _plot_path = os.path.join(_out_dir, "training_curve.png")
    _plt.savefig(_plot_path, dpi=100)
    _plt.close()
    print(f"Loss curve → {_plot_path}")
except Exception as _pe:
    print(f"Plot skipped: {_pe}")

print(f"\ntraining_loss = {metrics.get('train_loss', 'N/A')}")


In [ ]:
# ============================================================================
# 4F. FORMAT VALIDATION: 20 validation samples across DIFFERENT questions
# ============================================================================
import os, sys, json, random, torch
from collections import Counter, defaultdict

print("\n── Generating on 20 validation samples across DIFFERENT questions ──")
model.eval()

# ---- Locate val_sft.jsonl ------------------------------------------------
_data_dir = getattr(CFG, "data_dir", None) or os.environ.get("DATA_DIR", "/kaggle/working/data")
val_path = os.path.join(_data_dir, "val_sft.jsonl")
if not os.path.isfile(val_path):
    import glob as _gl
    _cands = sorted(_gl.glob("/kaggle/**/*val_sft.jsonl", recursive=True))
    if _cands:
        val_path = _cands[0]
print(f"Validation data path: {val_path}")

# ---- Load & filter rows --------------------------------------------------
val_by_question: dict = defaultdict(list)
_skipped = 0
with open(val_path, encoding="utf-8") as _f:
    for _line in _f:
        _line = _line.strip()
        if not _line:
            continue
        try:
            _row = json.loads(_line)
        except json.JSONDecodeError:
            _skipped += 1
            continue
        # GUARD: only keep rows that have the messages key
        if "messages" not in _row or not isinstance(_row["messages"], list) or len(_row["messages"]) < 3:
            _skipped += 1
            continue
        _qid = _row.get("question_id", "unknown")
        val_by_question[_qid].append(_row)

if _skipped:
    print(f"[WARN] Skipped {_skipped} malformed/incomplete rows.")

if not val_by_question:
    print("[WARN] No valid rows found in val_sft.jsonl. Skipping 4F.")
else:
    # Draw 20 samples balancing across different questions
    _rng_smoke = random.Random(getattr(CFG, "seed", 42))
    _q_pool = list(val_by_question.keys())
    _rng_smoke.shuffle(_q_pool)
    selected_samples = []

    while len(selected_samples) < 20 and _q_pool:
        _changed = False
        for _qid in list(_q_pool):
            if val_by_question[_qid]:
                selected_samples.append(val_by_question[_qid].pop(0))
                _changed = True
                if len(selected_samples) == 20:
                    break
            else:
                _q_pool.remove(_qid)
        if not _changed:
            break

    print(f"Selected {len(selected_samples)} samples across {len({s.get('question_id') for s in selected_samples})} questions.")

    valid_count = 0
    error_types = Counter()

    for i, row in enumerate(selected_samples, 1):
        msgs = row["messages"][:-1]  # drop the gold assistant turn
        user_content = row["messages"][1]["content"]

        # Parse rubric from the user message
        try:
            rubric_raw = user_content.split("## Rubric")[1].split("## Student")[0].strip()
            rubric = json.loads(rubric_raw)
        except Exception:
            rubric = []

        # Parse student answer from the user message
        try:
            stud_ans = user_content.split("```\n")[1].split("\n```")[0]
        except IndexError:
            stud_ans = ""

        qid = row.get("question_id", "?")

        try:
            text = tokenizer.apply_chat_template(
                msgs, tokenize=False, add_generation_prompt=True, enable_thinking=False,
            )
        except TypeError:
            text = tokenizer.apply_chat_template(
                msgs, tokenize=False, add_generation_prompt=True,
            )

        enc = tokenizer(text, return_tensors="pt").to(model.device)
        with torch.no_grad():
            out = model.generate(
                **enc, max_new_tokens=512, do_sample=False, pad_token_id=tokenizer.eos_token_id
            )
        raw = tokenizer.decode(out[0][enc.input_ids.shape[1]:], skip_special_tokens=True).strip()

        clean = raw
        if "```json" in clean:
            clean = clean.split("```json")[-1].split("```")[0].strip()
        elif "```" in clean:
            clean = clean.split("```")[1].strip()

        try:
            ok, errs = validate_grading_output(clean, rubric, stud_ans)
        except Exception as ve:
            ok, errs = False, [str(ve)]

        if ok:
            valid_count += 1
            status = "PARSE+VALID"
        else:
            status = "INVALID"
            for err in errs:
                err_cat = err.split(":")[1].strip() if ":" in err else err
                error_types[err_cat] += 1

        print(f"  [{i:2d}/20] ({qid:18s}) -> {status}")
        if not ok:
            print(f"         Error: {errs[:1]}")

    valid_rate = valid_count / max(len(selected_samples), 1)
    print("\n" + "="*60)
    print(f"Raw-output schema validity rate: {valid_count}/{len(selected_samples)} ({valid_rate:.1%})")
    print("Validator errors grouped by type:")
    if error_types:
        for etype, count in error_types.most_common():
            print(f"  \u2022 [{count}x] {etype}")
    else:
        print("  \u2022 None (all outputs strictly conformed to schema)")
    print("Note: Format verification only (proves output JSON structure works, not accuracy).")
    print("="*60 + "\n")


In [ ]:
# ============================================================================
# 4G. CLEANUP
# ============================================================================
import os, gc, torch

if "BEST_ADAPTER_PATH" not in globals():
    _out = getattr(CFG, "output_dir", "/kaggle/working/output")
    BEST_ADAPTER_PATH = os.path.join(_out, "best_adapter")

del trainer, model
gc.collect()
torch.cuda.empty_cache()
print(f"\n[PASS] NB2 complete.  Adapter → {BEST_ADAPTER_PATH}")
print(f"       Next → run NB3 (nb3_infer.py)\n")
print("""
WHAT TO CHECK NEXT
==================
1. training_curve.png: eval loss should decrease. If it is flat or rises,
   try more epochs or lower learning rate.
2. dropped_train in train_config.json: if > 10% of examples are dropped
   (too long), increase max_seq_len or shorten NB1 prompts.
3. ETA warning: if projected > 8 hours, reduce n_train in NB1 or increase
   gradient_accumulation_steps to reduce eval/save frequency.

KNOWN FAILURE MODES
===================
• NaN loss at step N → lower CFG.learning_rate to 1e-4, set max_grad_norm=0.1.
• "Response template not found" → chat template changed between transformers
  versions. Update RESPONSE_TEMPLATE to match the actual template separator.
• CUDA OOM → reduce per_device_train_batch_size or lora_r.
• Resume error after changing data/config → delete stale checkpoints in /kaggle/working/checkpoints/
  (a checkpoint from a different run cannot be resumed).
""")
